Run before everything

In [10]:
from ugot import ugot  # UGOT robot SDK
from IPython.display import clear_output  # Jupyter display helpers

# Create a robot instance and connect over the network.
got = ugot.UGOT()
got.initialize("192.168.88.1")  # <-- Check if this matches your robot's IP address

# Pre-load vision models onto the UGOT
# All models listed here will be available for the rest of the session.
got.load_models(
    ["color_recognition", "word_recognition", "line_recognition", "apriltag_qrcode"]
)

# Open UGOT camera for later image recognition
got.open_camera()

192.168.88.1:50051


Part2

In [ ]:
import time

# --- Configuration Constants ---
KP = 0.45          # Line following sensitivity
BASE_SPEED = 18    # Standard cruising speed
ZIGZAG_SPEED = 12  # Slower speed for searching to ensure "no wheels cross line"
SEARCH_TIMEOUT = 5 # Max seconds to zigzag before giving up

def follow_line_step(kp, base_speed):
    """Executes one step of proportional line following."""
    offset, line_type, x, y = got.get_single_track_total_info()
    if line_type == 0:
        return None
    
    # Proportional control for steering
    steering = int(offset * kp)
    # Slow down during sharper turns to prevent dropping the token (Rule C)
    speed_reduction = abs(int(steering * 0.2))
    current_speed = max(base_speed - speed_reduction, 10)
    
    got.mecanum_move_xyz(0, current_speed, steering)
    return line_type

def zigzag_search(direction):
    """
    Performs a directional zigzag to find the specific path.
    If 'LEFT', it starts by sweeping left. If 'RIGHT', it sweeps right.
    """
    print(f"Commencing Zigzag Search for: {direction}")
    
    # Determine search bias based on text recognition
    # Positive Z moves Left, Negative Z moves Right
    primary_z = 22 if direction == "LEFT" else -22
    secondary_z = -primary_z
    
    found = False
    start_search_time = time.time()
    
    # Increasing width sweep (0.4s, then 0.8s, then 1.2s)
    for i in range(1, 4):
        sweep_duration = 0.4 * i
        
        # 1. Sweep toward the intended path
        got.mecanum_move_xyz(0, ZIGZAG_SPEED, primary_z)
        sweep_start = time.time()
        while time.time() - sweep_start < sweep_duration:
            _, lt, _, _ = got.get_single_track_total_info()
            if lt != 0:
                found = True; break
            time.sleep(0.01)
        if found: break

        # 2. Sweep back across center to ensure we don't miss it
        got.mecanum_move_xyz(0, ZIGZAG_SPEED, secondary_z)
        sweep_start = time.time()
        while time.time() - sweep_start < (sweep_duration * 1.5):
            _, lt, _, _ = got.get_single_track_total_info()
            if lt != 0:
                found = True; break
            time.sleep(0.01)
        if found: break
        
        if time.time() - start_search_time > SEARCH_TIMEOUT:
            break

    if found:
        print("Path Acquired.")
        # Brief pause to stabilize before resuming full speed
        got.mecanum_stop()
        time.sleep(0.2)
    else:
        print("Search Failed.")

def part2run():
    print("Mission Started: Phase 1 (Approaching Fork)")
    got.set_track_recognition_line(0) 
    
    # --- STEP 1: Follow line until the gap/fork ---
    while True:
        status = follow_line_step(KP, BASE_SPEED)
        if status is None: # Line lost
            got.mecanum_stop()
            break
        time.sleep(0.01)

    # --- STEP 2: OCR Recognition (Fuzzy Match) ---
    print("Scanning for Text...")
    choice = ""
    while not choice:
        # Convert to string and uppercase for safety
        word = str(got.get_words_result()).upper()
        
        if any(p in word for p in ["LEFT", "LEF", "EFT"]):
            choice = "LEFT"
        elif any(p in word for p in ["RIGHT", "RIG", "GHT", "RGT"]):
            choice = "RIGHT"
            
        got.mecanum_stop()
        time.sleep(0.1)
    
    print(f"Confirmed Direction: {choice}")
    got.screen_display_background(6) # Visual feedback

    # --- STEP 3: Directional Transition ---
    # Move forward slightly to clear the 'dead zone' where the lines split
    got.mecanum_move_speed_times(direction=1, speed=15, times=0.5, unit=0)
    
    # Execute the Zigzag
    zigzag_search(choice)

    # --- STEP 4: Final Line Trace to Finish ---
    print("Phase 4: Final Path to Finish Line")
    while True:
        status = follow_line_step(KP, BASE_SPEED - 3)
        if status is None:
            # Check again to confirm it's not just a small gap
            time.sleep(0.3)
            _, final_lt, _, _ = got.get_single_track_total_info()
            if final_lt == 0:
                got.mecanum_stop()
                got.screen_print("FINISH")
                break
        time.sleep(0.01)

if __name__ == "__main__":
    try:
        part2run()
    except Exception as e:
        print(f"Error: {e}")
        got.mecanum_stop()

Mission Started: Phase 1 (Initial Following)
5
5
5
